НАЧИНАЕМ ML

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

race = pd.read_csv("../data/data_final/race_features.csv")

#на всякий защитились от дублей при старом merge
dup_cols = [c for c in ["quali_position_x", "quali_position_y"] if c in race.columns]
race = race.drop(columns=dup_cols)

print("Загружено:", race.shape)
race.head()

Загружено: (1397, 43)


,session_key,meeting_key,driver_number,position,number_of_laps,dnf,dns,dsq,gap_to_leader,points,...,team_form_5,driver_track_history,finished,driver_finish_rate,driver_experience,driver_speed_5,field_size,position_filled,is_rookie,quali_position
0,7953,1141,10,9.0,57.0,False,False,False,73.753,2.0,...,11.4,10.0,1,0.888889,0,321.2,20,9.0,1,20.0
1,7953,1141,55,4.0,57.0,False,False,False,48.052,12.0,...,11.4,10.0,1,0.888889,0,321.2,20,4.0,1,4.0
2,7953,1141,16,NaN,39.0,True,False,False,NaN,0.0,...,4.0,10.0,0,0.888889,0,321.2,20,20.0,1,3.0
3,7953,1141,27,15.0,56.0,False,False,False,+1 LAP,0.0,...,11.4,10.0,1,0.888889,0,321.2,20,15.0,1,10.0
4,7953,1141,20,13.0,56.0,False,False,False,+1 LAP,0.0,...,15.0,10.0,1,0.888889,0,321.2,20,13.0,1,17.0


Делим признаки на категориальные числовые и таргет, а так же не берем остальные признаки по типу position, position_filled, finished, dnf, dns, dsq, всё это - результат гонки. position_filled мы взяли целевой, а остальные это те же сведения о результате другими словами. Так же не берем best_time, gap_to_leader, points, best_lap, avg_lap, avg_st_speed, max_st_speed, avg_i1_speed, avg_i2_speed, laps_count, number_of_laps - все они измерены во время самой гонки. И так же выкидываем идентификаторы: session_key, meeting_key, driver_number, full_name, name_acronym, date_start, year, country_name - это номера сессий, имена пилотов, даты. 

In [4]:
num_features = ["quali_position", "driver_form_5", "driver_form_all", "team_form_5",
                "driver_track_history", "driver_finish_rate", "driver_experience",
                "driver_speed_5", "is_rookie", "field_size",
                "air_temp_mean", "track_temp_mean", "humidity_mean", "wind_speed_mean", "rainfall_max"]

cat_features = ["team_name", "circuit_short_name", "circuit_type"]

target = "position_filled"

print("Числовых:", len(num_features), "| категориальных:", len(cat_features))

Числовых: 15 | категориальных: 3


Делим хронологически - учим на 23-24, проверяем на 25

In [5]:
train = race[race["year"].isin([2023, 2024])].reset_index(drop=True)
test = race[race["year"] == 2025].reset_index(drop=True)

y_train = train[target]
y_test = test[target]

print("train:", train.shape, "гонок", train["session_key"].nunique())
print("test:", test.shape, "гонок", test["session_key"].nunique())

train: (918, 43) гонок 46
test: (479, 43) гонок 24


Так же кайфово и удобно сделаем функцию для метрик

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def print_metrics(name, y_true, y_pred):
    print(name)
    print("MAE ", round(mean_absolute_error(y_true, y_pred), 3))
    print("RMSE", round(mean_squared_error(y_true, y_pred) ** 0.5, 3))
    print("R2  ", round(r2_score(y_true, y_pred), 3))
    print()

Кодируем категориальные признаки

In [7]:

X_all = pd.get_dummies(race[num_features + cat_features], columns=cat_features, drop_first=True)

X_train = X_all[race["year"].isin([2023, 2024])].reset_index(drop=True)
X_test = X_all[race["year"] == 2025].reset_index(drop=True)

print("Признаков после get_dummies:", X_all.shape[1])

Признаков после get_dummies: 49


Дальше масштабируем StandardScaler, обучая его строго на train

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

Обучем линейную регрессию

In [11]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = lr.predict(X_test_sc)

print_metrics("LINEAR REGRESSION", y_test, pred_lr)

LINEAR REGRESSION
MAE  3.455
RMSE 4.523
R2   0.423



Мы получили R2 равной 0.42, что для линейной модели оч даже неплохо и мы обьясняем 42 процента разброса. МАЕ 3.46 значит в среднем мы ошибаемся на 3-4 позиции, что в целом неплохо для прикидки но все еще мало для прогноза

Теперь попробуем отобрать 15 признаков с наибольшей связью и посмотрим что выйдетю

In [12]:
from sklearn.feature_selection import SelectKBest, f_regression

sb = SelectKBest(f_regression, k=15)
X_train_kbest = sb.fit_transform(X_train_sc, y_train)
X_test_kbest = sb.transform(X_test_sc)

lr = LinearRegression()
lr.fit(X_train_kbest, y_train)
pred_kbest = lr.predict(X_test_kbest)

print_metrics("LINREG + SelectKBest (k=15)", y_test, pred_kbest)
print("Отобраны:", list(X_train.columns[sb.get_support()]))

LINREG + SelectKBest (k=15)
MAE  3.355
RMSE 4.466
R2   0.437

Отобраны: ['quali_position', 'driver_form_5', 'driver_form_all', 'team_form_5', 'driver_track_history', 'driver_finish_rate', 'driver_speed_5', 'team_name_Ferrari', 'team_name_Haas F1 Team', 'team_name_Kick Sauber', 'team_name_McLaren', 'team_name_Mercedes', 'team_name_Racing Bulls', 'team_name_Red Bull Racing', 'team_name_Williams']


Р2 немного выросла и МАЕ уменьшилось, значит отбор признаков работает вполне качественно и убирает лишнее из OHE.

Проверим что будем при PCA - сожмем признаки в 10 компонент и посмотрим сколько дисперсии они сохраняют.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca = pca.transform(X_test_sc)

lr = LinearRegression()
lr.fit(X_train_pca, y_train)
pred_pca = lr.predict(X_test_pca)

print_metrics("LINREG + PCA (n=10)", y_test, pred_pca)

LINREG + PCA (n=10)
MAE  3.704
RMSE 4.781
R2   0.355

Объяснённая дисперсия: 0.451


PCA сделал только хуже и в целом это не с проста тк PCA смешивает все признаки в компоненты и наш самые сильный предикатор размыается по ним вместе с шумом, теряя свою силу

Заведем временную кросс-валидацию - она нам будет нужна для всех подборов

In [15]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

KNN с подбором. Перебираем число соседей, тип весов и метрику (p=1 манхэттен, p=2 евклид) 

In [16]:
from sklearn.neighbors import KNeighborsRegressor

param_knn = {
    "n_neighbors": [5, 10, 15, 20, 30],
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}

gs_knn = GridSearchCV(KNeighborsRegressor(), param_knn, cv=tscv,
                      scoring="neg_mean_absolute_error", n_jobs=-1)
gs_knn.fit(X_train_sc, y_train)

print("Лучшие параметры kNN:", gs_knn.best_params_)
print_metrics("kNN (tuned)", y_test, gs_knn.predict(X_test_sc))

Лучшие параметры kNN: {'n_neighbors': 20, 'p': 1, 'weights': 'distance'}
kNN (tuned)
MAE  3.825
RMSE 4.786
R2   0.354



Результат стал только хуже, чем у линейной регрессии - скип

SVR с подбором. Перебираем C (жёсткость), gamma (ширина ядра) и epsilon (зона нечувствительности)

In [17]:
from sklearn.svm import SVR

param_svr = {
    "C": [1, 10, 50],
    "gamma": ["scale", 0.01, 0.05],
    "epsilon": [0.1, 0.5, 1.0],
}

gs_svr = GridSearchCV(SVR(), param_svr, cv=tscv,
                      scoring="neg_mean_absolute_error", n_jobs=-1)
gs_svr.fit(X_train_sc, y_train)

print("Лучшие параметры SVR:", gs_svr.best_params_)
print_metrics("SVR (tuned)", y_test, gs_svr.predict(X_test_sc))

Лучшие параметры SVR: {'C': 10, 'epsilon': 0.5, 'gamma': 0.01}
SVR (tuned)
MAE  3.441
RMSE 4.689
R2   0.38



Так же ухже лин рега - скип

Дерево с подбором глубины и минимальным колвом обьектов в листе

In [18]:
from sklearn.tree import DecisionTreeRegressor

param_tree = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_leaf": [1, 5, 10, 20],
}

gs_tree = GridSearchCV(DecisionTreeRegressor(random_state=13), param_tree, cv=tscv,
                       scoring="neg_mean_absolute_error", n_jobs=-1)
gs_tree.fit(X_train, y_train)

print("Лучшие параметры дерева:", gs_tree.best_params_)
print_metrics("Decision Tree (tuned)", y_test, gs_tree.predict(X_test))

Лучшие параметры дерева: {'max_depth': 3, 'min_samples_leaf': 10}
Decision Tree (tuned)
MAE  3.439
RMSE 4.505
R2   0.427



В целом неплохо, но почти так же как и линейная регрессия

Случайный лес с подбором гиперпараметров

In [19]:
from sklearn.ensemble import RandomForestRegressor

param_rf = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 5, 10],
}

gs_rf = GridSearchCV(RandomForestRegressor(random_state=13, n_jobs=-1), param_rf, cv=tscv,
                     scoring="neg_mean_absolute_error", n_jobs=-1)
gs_rf.fit(X_train, y_train)

print("Лучшие параметры леса:", gs_rf.best_params_)
print_metrics("Random Forest (tuned)", y_test, gs_rf.predict(X_test))

Лучшие параметры леса: {'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 100}
Random Forest (tuned)
MAE  3.42
RMSE 4.456
R2   0.44



Вот это другое дело, теперь мы можем обьяснить 44 процента разброса - что в целом уже не так плохо 

Теперь попробуем модели следующей природы: CatBoost с дефолтными параметрами

In [27]:
from catboost import CatBoostRegressor

cat_default = CatBoostRegressor(random_seed=13, verbose=0)
cat_default.fit(X_train, y_train)
print_metrics("CatBoost (default)", y_test, cat_default.predict(X_test))

CatBoost (default)
MAE  3.672
RMSE 4.73
R2   0.369



Кэтбуст без подбора параметров дал результат хуже чем у нас был, поэтому попробуем новый инструмент - Optuna, который предлагает параметры, мы считаем средний MAE на временной кросс-валидации и минимизируем его. У CatBoost ключевые параметры — число деревьев iterations, скорость обучения, глубина depth и регуляризация l2_leaf_reg:

In [28]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 3, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_seed": 13,
        "verbose": 0,
    }
    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=tscv,
                             scoring="neg_mean_absolute_error")
    return -scores.mean()

/Users/timofej/Desktop/gp-iad/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используем функцию так как в Optuna больше никак по-другому нельзя 

In [29]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=25)

print("Лучшие параметры:", study.best_params)

Лучшие параметры: {'iterations': 530, 'learning_rate': 0.014391140774709392, 'depth': 3, 'l2_leaf_reg': 6.86741379181175}


Теперь финальная модель на лучших параметрах

In [30]:
cat_best = CatBoostRegressor(**study.best_params, random_seed=13, verbose=0)
cat_best.fit(X_train, y_train)
print_metrics("CatBoost (Optuna)", y_test, cat_best.predict(X_test))

CatBoost (Optuna)
MAE  3.374
RMSE 4.449
R2   0.442



Бизнес-метрики — их считаем для любой модели, поэтому оформим как функцию оценщик. Внутри каждой гонки ранжируем предсказания в порядок 1–20 и считаем человеческие метрики.

In [31]:
from scipy.stats import spearmanr

def business_metrics(test_df, order_col):
    win = podium = top10 = 0
    spear = []
    for sk, g in test_df.groupby("session_key"):
        real_win = g.loc[g["position"] == 1, "driver_number"]
        pred_win = g.loc[g[order_col] == 1, "driver_number"]
        if len(real_win) and len(pred_win) and real_win.iloc[0] == pred_win.iloc[0]:
            win += 1
        podium += len(set(g.loc[g["position"] <= 3, "driver_number"]) &
                      set(g.loc[g[order_col] <= 3, "driver_number"]))
        top10 += len(set(g.loc[g["position"] <= 10, "driver_number"]) &
                     set(g.loc[g[order_col] <= 10, "driver_number"]))
        fg = g.dropna(subset=["position"])
        if len(fg) > 2:
            spear.append(spearmanr(fg["position"], fg[order_col]).correlation)
    n = test_df["session_key"].nunique()
    print("Угадан победитель:", win, "/", n)
    print("Подиум (из 3):", round(podium / n, 2))
    print("Топ-10 (из 10):", round(top10 / n, 2))
    print("Spearman:", round(np.mean(spear), 3))
    print()

Считаем бизнес-метрики для регрессора. Берём наш настроенный CatBoost, ранжируем предсказания внутри каждой гонки в порядок и оцениваем:

In [32]:
test_eval = test.copy()
test_eval["pred"] = cat_best.predict(X_test)
test_eval["order"] = test_eval.groupby("session_key")["pred"].rank(method="first")

print("РЕГРЕССОР (CatBoost):")
business_metrics(test_eval, "order")

РЕГРЕССОР (CatBoost):
Угадан победитель: 12 / 24
Подиум (из 3): 2.21
Топ-10 (из 10): 7.71
Spearman: 0.757



Теперь ранжировщик, его идея в том, что модель учится не на абсолютной позиции, а на порядке пилотов внутри гонки. Для этого нужны группы (каждая гонка это отдельная группа через group_id) и релевантность (чем выше финиш, тем больше - берём field_size - position_filled). Данные надо отсортировать по session_key, чтобы группы шли подряд.

In [33]:
from catboost import CatBoostRanker, Pool

race_rank = race.copy()
race_rank["relevance"] = race_rank["field_size"] - race_rank["position_filled"]

train_r = race_rank[race_rank["year"].isin([2023, 2024])].sort_values("session_key")
test_r = race_rank[race_rank["year"] == 2025].sort_values("session_key").reset_index(drop=True)

X_train_r = pd.get_dummies(train_r[num_features + cat_features], columns=cat_features, drop_first=True)
X_test_r = pd.get_dummies(test_r[num_features + cat_features], columns=cat_features, drop_first=True)
X_test_r = X_test_r.reindex(columns=X_train_r.columns, fill_value=0)

train_pool = Pool(X_train_r, label=train_r["relevance"], group_id=train_r["session_key"])

Обучаем и оцениваем теми же бизнес-метриками (порядок строим по убыванию скора, выше скор значит выше финиш):

In [34]:
ranker = CatBoostRanker(iterations=400, learning_rate=0.05, depth=4,
                        loss_function="YetiRank", random_seed=13, verbose=0)
ranker.fit(train_pool)

test_r["score"] = ranker.predict(X_test_r)
test_r["order"] = test_r.groupby("session_key")["score"].rank(method="first", ascending=False)

print("РАНЖИРОВЩИК (CatBoostRanker):")
business_metrics(test_r, "order")

РАНЖИРОВЩИК (CatBoostRanker):
Угадан победитель: 17 / 24
Подиум (из 3): 2.04
Топ-10 (из 10): 7.62
Spearman: 0.721



Теперь сравним все модели. Для этого копирую прошлую функцию оценщика и чуть меняю выход

In [35]:
def eval_order(df, order_col="order"):
    win = pod = top10 = 0
    spear = []
    for sk, g in df.groupby("session_key"):
        rw = g.loc[g["position"] == 1, "driver_number"]
        pw = g.loc[g[order_col] == 1, "driver_number"]
        if len(rw) and len(pw) and rw.iloc[0] == pw.iloc[0]:
            win += 1
        pod += len(set(g.loc[g["position"] <= 3, "driver_number"]) &
                   set(g.loc[g[order_col] <= 3, "driver_number"]))
        top10 += len(set(g.loc[g["position"] <= 10, "driver_number"]) &
                     set(g.loc[g[order_col] <= 10, "driver_number"]))
        fg = g.dropna(subset=["position"])
        if len(fg) > 2:
            spear.append(spearmanr(fg["position"], fg[order_col]).correlation)
    n = df["session_key"].nunique()
    return win, round(pod / n, 2), round(top10 / n, 2), round(np.mean(spear), 3)

Собираем предсказания всех моделей-регрессоров (используем уже обученные объекты) и считаем обе группы метрик

In [37]:
lr_full = LinearRegression().fit(X_train_sc, y_train)

preds = {
    "LinearRegression": lr_full.predict(X_test_sc),
    "kNN": gs_knn.predict(X_test_sc),
    "SVR": gs_svr.predict(X_test_sc),
    "DecisionTree": gs_tree.predict(X_test),
    "RandomForest": gs_rf.predict(X_test),
    "CatBoost": cat_best.predict(X_test),
}

rows = []
for name, pred in preds.items():
    d = test.copy()
    d["pred"] = pred
    d["order"] = d.groupby("session_key")["pred"].rank(method="first")
    win, pod, top10, sp = eval_order(d)
    rows.append([name,
                 round(mean_absolute_error(y_test, pred), 2),
                 round(mean_squared_error(y_test, pred) ** 0.5, 2),
                 round(r2_score(y_test, pred), 3),
                 f"{win}/24", pod, top10, sp])

Добавляем ранжировщик отдельной строкой тк у него нет р2 и мае, он работает с порядками а не позициями:

In [38]:
win, pod, top10, sp = eval_order(test_r)
rows.append(["CatBoostRanker", "-", "-", "-", f"{win}/24", pod, top10, sp])

results_table = pd.DataFrame(rows, columns=[
    "Модель", "MAE", "RMSE", "R2", "Победитель", "Подиум/3", "Топ-10/10", "Spearman"
])
results_table

,Модель,MAE,RMSE,R2,Победитель,Подиум/3,Топ-10/10,Spearman
0,LinearRegression,3.46,4.52,0.423,13/24,2.17,7.62,0.748
1,kNN,3.82,4.79,0.354,11/24,1.92,7.29,0.670
2,SVR,3.44,4.69,0.38,10/24,2.04,7.62,0.735
3,DecisionTree,3.44,4.51,0.427,7/24,1.79,7.25,0.708
4,RandomForest,3.42,4.46,0.44,9/24,2.12,7.62,0.742
5,CatBoost,3.37,4.45,0.442,12/24,2.21,7.71,0.757
6,CatBoostRanker,-,-,-,17/24,2.04,7.62,0.721


Выводы:

Ансамбли лучшие по техническим метрикам. RandomForest R2 = 0.442 и CatBoost 0.441 делят первое место, у CatBoost при этом самая низкая ошибка в позициях. Они ловят нелинейности и взаимодействия признаков, которых линейная модель не видит, и при этом не переобучаются. 

Методы на расстояниях проигрывают. kNN 0.354 и SVR 0.380 заметно слабее, им тяжело в пространстве из множества разреженных one-hot признаков, плюс гонка слишком шумная для геометрической близости.

Линейная модель на удивление крепкая. R2 0.423 почти догоняет ансамбли. Значит зависимость финиша от наших признаков в основном близка к линейной, а нелинейности добавляют лишь пару процентов.

Общий вывод: Лучшая рабочая модель это CatBoost, а если бизнесу важен именно порядок то CatBoostRanker. Все сильные модели упёрлись в R2 0.44, это потолок данных из-за непредсказуемых сходов, а не предел моделей.